In [1]:
import os

In [2]:
%pwd  # It returns the current working path

'd:\\AI_project\\Deep_Learning\\research'

In [3]:
os.chdir("../")  # It is on main folder now we have changed the path

In [4]:
%pwd # For accessing the all files we have changed the path

'd:\\AI_project\\Deep_Learning'

In [5]:
# 3. Update Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class  DataIngestionConfig:
    root_dir : Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path

#### 4. Update Configuration manager


In [6]:
# 4. Update Configuration manager
from src.cnnClassifier.constants import *

In [7]:
from src.cnnClassifier.utils.common import read_yaml,create_directories  # read_yaml :will read all the files yaml files  .....create_directories is used for creating directories


In [8]:
%pwd

'd:\\AI_project\\Deep_Learning'

In [9]:
class ConfigurationManager:
   def __init__(
       self,
      config_filepath=Path('config/config.yaml'),
      params_filepath=Path('params.yaml')):

       self.config = read_yaml(config_filepath)
       self.params = read_yaml(params_filepath)


       create_directories([self.config.artifacts_root])


   def get_data_ingestion_config(self) -> DataIngestionConfig:
       config = self.config.data_ingestion


       create_directories([config.root_dir])


       data_ingestion_config = DataIngestionConfig(
           root_dir=config.root_dir,
           source_URL=config.source_URL,
           local_data_file=config.local_data_file,
           unzip_dir=config.unzip_dir
       )


       return data_ingestion_config


#### Update Components

In [10]:
import zipfile
import gdown
from src.cnnClassifier import logger
from src.cnnClassifier.utils.common import get_size

In [11]:
class DataIngestion:
   def __init__(self, config: DataIngestionConfig):  # We assigning default value
       self.config = config


  
   def download_file(self)-> str:  # defining that the values are in string
       '''
       Fetch data from the url
       '''


       try:
           dataset_url = self.config.source_URL
           zip_download_dir = self.config.local_data_file
           os.makedirs("artifacts/data_ingestion", exist_ok=True)  # Creating a folder of artifacts
           logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")


           file_id = dataset_url.split("/")[-2]  # Spilting the data to get file_id (after splitting the link it will be in list that why we are wrtting -2)
           prefix = 'https://drive.google.com/uc?/export=download&id='
           gdown.download(prefix+file_id,zip_download_dir)


           logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")


       except Exception as e:
           raise e
      
  


   def extract_zip_file(self):  # Code for unzipping the folder
       """
       zip_file_path: str
       Extracts the zip file into the data directory
       Function returns None
       """
       unzip_path = self.config.unzip_dir
       os.makedirs(unzip_path, exist_ok=True)
       with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
           zip_ref.extractall(unzip_path)


#### Pipeline


In [12]:
try:
   config = ConfigurationManager()
   data_ingestion_config = config.get_data_ingestion_config()
   data_ingestion = DataIngestion(config=data_ingestion_config)
   data_ingestion.download_file()
   data_ingestion.extract_zip_file()
except Exception as e:
   raise e

[2026-01-05 12:40:00,366: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-05 12:40:00,371: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-05 12:40:00,373: INFO: common: created directory at: artifacts]
[2026-01-05 12:40:00,374: INFO: common: created directory at: artifacts/data_ingestion]
[2026-01-05 12:40:00,378: INFO: 564314066: Downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=2139fac0-5011-4e0b-abd7-be9636d30255
To: d:\AI_project\Deep_Learning\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:05<00:00, 11.4MB/s]

[2026-01-05 12:40:08,324: INFO: 564314066: Downloaded data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]
